# pyquant quickstart

End-to-end minimal demo: tweets → analysis → price → signals.
Runs in mock mode if no API keys are set.

In [ ]:
import pandas as pd
from datetime import datetime

from pyquant import config, db, x_client, openai_client, yfinance_client, analyze, price, signals

settings = config.get_settings()
engine = db.get_engine(settings.DB_URL)
db.create_tables(engine)

conn = engine.connect()
dialect = engine.dialect.name
print("DB:", settings.DB_URL)
print("Dialect:", dialect)
print("HAS_X:", settings.HAS_X, "HAS_OPENAI:", settings.HAS_OPENAI)

## Fetch tweets

In [ ]:
handles = ['alice', 'bob']  # replace with real handles if you have a token
since = None  # e.g., '2024-01-01T00:00:00Z'
until = None
tweets = x_client.get_tweets(handles, since=since, until=until)
if not settings.HAS_X:
    print('Mock mode: using sample tweets')

df_tweets = pd.DataFrame(tweets)
display(df_tweets)

# Write to DB (INSERT IGNORE semantics via helper)
for t in tweets:
    db.insert_tweet(conn, user_handle=t['user_handle'], tweet_id=t['tweet_id'], text_=t['text'], created_at=t['created_at'])

print(f'Inserted tweets: {len(tweets)}')

## Analyze with OpenAI (or mock)

In [ ]:
analyses = openai_client.analyze_tweets(tweets)
if not settings.HAS_OPENAI:
    print('Mock analysis: deterministic coin/stance output')

rows = []
for a in analyses:
    for coin in a['coins']:
        rows.append({
            'tweet_id': a['tweet_id'],
            'coin_symbol': coin,
            'stance': a['stance'],
            'rationale': a['rationale'],
            'confidence': a['confidence'],
            'model': a['model'],
            'created_at': datetime.utcnow().isoformat() + 'Z'
        })

df_analysis = pd.DataFrame(rows)
display(df_analysis)

for r in rows:
    db.insert_analysis(conn, **r)

print(f'Inserted analysis rows: {len(rows)}')

## Price windows

In [ ]:
price_rows = []
for r in rows:
    # Use tweet created time as t0; if missing, now
    t0 = pd.to_datetime(df_tweets.set_index('tweet_id').loc[r['tweet_id'], 'created_at'], utc=True, errors='coerce')
    if pd.isna(t0):
        t0 = pd.Timestamp.utcnow()
    t0 = t0.to_pydatetime()
    pw = yfinance_client.get_price_window(r['coin_symbol'] + '-USD', t0, horizon_days=7)
    if pw is None:
        continue
    price_rows.append({
        'coin_symbol': r['coin_symbol'],
        't0': t0.isoformat() + 'Z',
        't1': pw['t1'].isoformat() + 'Z',
        'max_price': pw['max_price'],
        'min_price': pw['min_price'],
        'vol': pw['vol'],
    })

df_prices = pd.DataFrame(price_rows)
display(df_prices)

for pr in price_rows:
    db.insert_price_window(conn, **pr)

print(f'Inserted price windows: {len(price_rows)}')

## Signals

In [ ]:
# Merge analysis with prices on coin symbol; fallback momentum to 0 if price missing
df = df_analysis.copy()
if 'df_prices' in globals() and not df_prices.empty:
    df = df.merge(df_prices[['coin_symbol', 't0', 't1', 'max_price', 'min_price', 'vol']], on='coin_symbol', how='left')
else:
    df['t0'] = pd.Timestamp.utcnow().isoformat() + 'Z'
    df['t1'] = df['t0']
    df['max_price'] = None
    df['min_price'] = None
    df['vol'] = 0.0

def _momentum_row(row):
    try:
        # if we fetched prices, use midpoint as t0 price proxy
        if pd.notna(row['max_price']) and pd.notna(row['min_price']):
            t0p = (row['max_price'] + row['min_price']) / 2.0
            return price.momentum(t0p, row['max_price'], row['min_price'])
    except Exception:
        pass
    return 0.0

df['stance_dir'] = df['stance'].map(analyze.stance_to_dir)
df['momentum'] = df.apply(_momentum_row, axis=1)
df['signal_score'] = df.apply(lambda r: signals.score_signal(r['confidence'], r['stance_dir'], r['momentum']), axis=1)
df['horizon_days'] = 7
df['comment'] = df.apply(lambda r: f"{r['coin_symbol']} {r['stance']}", axis=1)

display(df[['tweet_id','coin_symbol','stance','confidence','momentum','signal_score']].sort_values('signal_score', ascending=False))

for _, r in df.iterrows():
    db.insert_signal(
        conn,
        tweet_id=str(r['tweet_id']),
        coin_symbol=str(r['coin_symbol']),
        signal_score=float(r['signal_score']),
        horizon_days=int(r['horizon_days']),
        comment=str(r['comment']),
        created_at=datetime.utcnow().isoformat() + 'Z'
    )
print(f'Inserted signals: {len(df)}')

## Helper: run end-to-end

In [ ]:
def run_pipeline(handles, since=None, until=None, horizon_days=7):
    tweets = x_client.get_tweets(handles, since=since, until=until)
    for t in tweets:
        db.insert_tweet(conn, user_handle=t['user_handle'], tweet_id=t['tweet_id'], text_=t['text'], created_at=t['created_at'])
    analyses = openai_client.analyze_tweets(tweets)
    rows = []
    for a in analyses:
        for coin in a['coins']:
            r = {
                'tweet_id': a['tweet_id'],
                'coin_symbol': coin,
                'stance': a['stance'],
                'rationale': a['rationale'],
                'confidence': a['confidence'],
                'model': a['model'],
                'created_at': datetime.utcnow().isoformat() + 'Z'
            }
            db.insert_analysis(conn, **r)
            rows.append(r)
    price_rows = []
    df_tweets = pd.DataFrame(tweets).set_index('tweet_id')
    for r in rows:
        t0 = pd.to_datetime(df_tweets.loc[r['tweet_id'], 'created_at'], utc=True, errors='coerce')
        if pd.isna(t0):
            t0 = pd.Timestamp.utcnow()
        t0 = t0.to_pydatetime()
        pw = yfinance_client.get_price_window(r['coin_symbol'] + '-USD', t0, horizon_days=horizon_days)
        if pw:
            pr = {
                'coin_symbol': r['coin_symbol'],
                't0': t0.isoformat() + 'Z',
                't1': pw['t1'].isoformat() + 'Z',
                'max_price': pw['max_price'],
                'min_price': pw['min_price'],
                'vol': pw['vol'],
            }
            db.insert_price_window(conn, **pr)
            price_rows.append(pr)
    df = pd.DataFrame(rows)
    if price_rows:
        dfp = pd.DataFrame(price_rows)
        df = df.merge(dfp[['coin_symbol','max_price','min_price']], on='coin_symbol', how='left')
    df['stance_dir'] = df['stance'].map(analyze.stance_to_dir)
    def _mom(r):
        if 'max_price' in r and 'min_price' in r and pd.notna(r['max_price']) and pd.notna(r['min_price']):
            t0p = (r['max_price'] + r['min_price'])/2.0
            return price.momentum(t0p, r['max_price'], r['min_price'])
        return 0.0
    df['momentum'] = df.apply(_mom, axis=1)
    df['signal_score'] = df.apply(lambda r: signals.score_signal(r['confidence'], r['stance_dir'], r['momentum']), axis=1)
    df['horizon_days'] = horizon_days
    df['comment'] = df.apply(lambda r: f"{r['coin_symbol']} {r['stance']}", axis=1)
    for _, r in df.iterrows():
        db.insert_signal(conn, tweet_id=str(r['tweet_id']), coin_symbol=str(r['coin_symbol']), signal_score=float(r['signal_score']), horizon_days=int(r['horizon_days']), comment=str(r['comment']), created_at=datetime.utcnow().isoformat() + 'Z')
    return df.sort_values('signal_score', ascending=False)

# Example usage
# run_pipeline(['alice','bob'])